<a href="https://colab.research.google.com/github/mkobycheva/recommendation-system/blob/a-b-test/notebooks/a_b_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!test -d /content/recommendation-system/.git \
  || git clone -b als https://github.com/mkobycheva/recommendation-system.git /content/recommendation-system

%cd /content/recommendation-system
!git fetch origin a-b-test
!git checkout a-b-test

import sys
sys.path.insert(0, '.')

from google.colab import drive
drive.mount('/content/drive')

Cloning into '/content/recommendation-system'...
remote: Enumerating objects: 561, done.
remote: Counting objects: 100% (159/159), done.
remote: Compressing objects: 100% (126/126), done.
remote: Total 561 (delta 81), reused 68 (delta 33), pack-reused 402 (from 1)
Receiving objects: 100% (561/561), 871.99 KiB | 17.44 MiB/s, done.
Resolving deltas: 100% (310/310), done.
/content/recommendation-system
From https://github.com/mkobycheva/recommendation-system
 * branch            a-b-test   -> FETCH_HEAD
Branch 'a-b-test' set up to track remote branch 'a-b-test' from 'origin'.
Switched to a new branch 'a-b-test'
Mounted at /content/drive


In [2]:
!pip install -q anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 956.9/956.9 kB 42.7 MB/s eta 0:00:00


In [3]:
import os
import numpy as np
import pandas as pd
import json
import time
import random
from concurrent.futures import ThreadPoolExecutor, as_completed
from anthropic import Anthropic, RateLimitError, APIConnectionError, InternalServerError, APITimeoutError

from src.data.build_matrix import add_domain_item_ids

In [43]:
from google.colab import userdata
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")

In [44]:
key = os.environ.get("ANTHROPIC_API_KEY")
print(f"key present: {key is not None}, length: {len(key) if key else 0}, starts with: {key[:7] if key else None}")

key present: True, length: 73, starts with: sk-or-v


## Getting the data

Historical

In [28]:
DATA_DIR = '/content/drive/MyDrive/recsys-data'

books_train = pd.read_csv(f'{DATA_DIR}/books_train.csv')
books_valid = pd.read_csv(f'{DATA_DIR}/books_valid.csv')
books_test = pd.read_csv(f'{DATA_DIR}/books_test.csv')

movies_train = pd.read_csv(f'{DATA_DIR}/movies_train.csv')
movies_valid = pd.read_csv(f'{DATA_DIR}/movies_valid.csv')
movies_test = pd.read_csv(f'{DATA_DIR}/movies_test.csv')

In [29]:
books_train = add_domain_item_ids(books_train, 'books')
books_valid = add_domain_item_ids(books_valid, 'books')
books_test = add_domain_item_ids(books_test, 'books')

movies_train = add_domain_item_ids(movies_train, 'movies')
movies_valid = add_domain_item_ids(movies_valid, 'movies')
movies_test = add_domain_item_ids(movies_test, 'movies')

books_train.head(5)

,user_id,parent_asin,rating,timestamp,domain,item_id
0,AFSKPY37N3C43SOI5IEXEK5JSIYA,0061713244,5.0,1224796701000,books,books::0061713244
1,AFSKPY37N3C43SOI5IEXEK5JSIYA,0871139634,5.0,1227669792000,books,books::0871139634
2,AFSKPY37N3C43SOI5IEXEK5JSIYA,0385521685,5.0,1231716230000,books,books::0385521685
3,AFSKPY37N3C43SOI5IEXEK5JSIYA,0848732634,3.0,1236304137000,books,books::0848732634
4,AFSKPY37N3C43SOI5IEXEK5JSIYA,0151014981,3.0,1245335041000,books,books::0151014981


In [30]:
books_meta = pd.read_csv(f'{DATA_DIR}/books_meta.csv')
movies_meta = pd.read_csv(f'{DATA_DIR}/movies_meta.csv')

books_title = books_meta.set_index('parent_asin')['title'].to_dict()
movies_title = movies_meta.set_index('parent_asin')['title'].to_dict()

def title_for(item_id):
    domain, asin = item_id.split('::', 1)
    lookup = books_title if domain == 'books' else movies_title
    return lookup.get(asin, '(title not found)')

ALS Recommendations

In [45]:
with open(f'{DATA_DIR}/als_books_recs_bulk.json') as f:
    als_books_recs = json.load(f)

with open(f'{DATA_DIR}/als_movies_recs_bulk.json') as f:
    als_movies_recs = json.load(f)

print(f"{len(als_books_recs)} book users, {len(als_movies_recs)} movie users")
als_books_recs[0]

5000 book users, 5000 movie users


{'user_id': 'AFSKPY37N3C43SOI5IEXEK5JSIYA',
 'domain': 'books',
 'recommended_item_ids': ['books::0345512146',
  'books::0618968415',
  'books::0805097252',
  'books::0307587606',
  'books::0393062333',
  'books::0061662585',
  'books::1607477009',
  'books::1400004586',
  'books::0307272222',
  'books::0061233331'],
 'recommended_titles': ["Twilight at the World of Tomorrow: Genius, Madness, Murder, and the 1939 World's Fair on the Brink of War",
  'The Big Burn: Teddy Roosevelt and the Fire That Saved America',
  'My Lunches with Orson: Conversations between Henry Jaglom and Orson Welles',
  'American History Revised: 200 Startling Facts That Never Made It into the Textbooks',
  'Yogi Berra: Eternal Yankee',
  "Inventing George Washington: America's Founder, in Myth and Memory",
  "Eighty Is Not Enough: One Actor's Journey through American Entertainment",
  "Fodor's New York City 2011 (Full-color Travel Guide)",
  'Dearie: The Remarkable Life of Julia Child',
  'The Long Way Home: An

BERT Recommendations

In [46]:
with open(f'{DATA_DIR}/bert_books_recs_bulk.json') as f:
    bert_books_recs = json.load(f)

with open(f'{DATA_DIR}/bert_movies_recs_bulk.json') as f:
    bert_movies_recs = json.load(f)

print(f"{len(bert_books_recs)} book users, {len(bert_movies_recs)} movie users")
bert_books_recs[0]

5000 book users, 5000 movie users


{'user_id': 'AFZUK3MTBIBEDQOPAK3OATUOUKLA',
 'domain': 'books',
 'recommended_item_ids': ['books::160774273X',
  'books::1616770759',
  'books::1945256567',
  'books::B072BLVM83',
  'books::1446307352',
  'books::1948209004',
  'books::132885213X',
  'books::059343451X',
  'books::1939754984',
  'books::1101885955'],
 'recommended_titles': ['Flour Water Salt Yeast: The Fundamentals of Artisan Bread and Pizza [A Cookbook]',
  'Piano Adventures - Lesson Book - Primer Level',
  "Cook It in Your Dutch Oven: 150 Foolproof Recipes Tailor-Made for Your Kitchen's Most Versatile Pot",
  'Educated: A Memoir',
  'DIY Watercolor Flowers: The beginner’s guide to flower painting for journal pages, handmade stationery and more (DIY Watercolor, 1)',
  "CreateSpace Classics Lettering and Modern Calligraphy: A Beginner's Guide: Learn Hand Lettering and Brush Lettering",
  'Good Night, Little Blue Truck',
  'The Bench',
  'My First Toddler Coloring Book: Fun with Numbers, Letters, Shapes, Colors, and Ani

## ALS: Building user profiles

In [32]:
random.seed(676767)  # reproducible sample

MIN_HISTORY_ITEMS = 3   # need at least this many valid historical items to build a profile
MAX_HISTORY_ITEMS = 20  # cap how much history we send to the agent (cost/context control)
N_USERS_PER_DOMAIN = 250


def has_valid_recs(entry):
    """True if a recs_bulk entry has clean, non-null recommendation data."""
    ids = entry.get('recommended_item_ids')
    titles = entry.get('recommended_titles')
    if not ids or not titles or len(ids) != len(titles):
        return False
    if any(pd.isna(x) or x in (None, '') for x in ids):
        return False
    if any(pd.isna(x) or x in (None, '') for x in titles):
        return False
    return True


def get_valid_history(group, min_items=MIN_HISTORY_ITEMS, max_items=MAX_HISTORY_ITEMS):
    """Most-recent-first (title, rating) history for one user's train rows.
    Drops rows with missing rating/item_id or an unresolved title.
    Returns None if too few valid items remain."""
    rows = group.dropna(subset=['item_id', 'rating']).sort_values('timestamp', ascending=False)

    history = []
    for _, r in rows.iterrows():
        title = title_for(r['item_id'])
        if title == '(title not found)':
            continue
        history.append({'title': title, 'rating': float(r['rating'])})
        if len(history) >= max_items:
            break

    return history if len(history) >= min_items else None


def sample_valid_users(recs, train_df, domain, n=N_USERS_PER_DOMAIN):
    """Randomly sample n users who have both clean recs and enough clean history."""
    # group once up front instead of re-filtering train_df per user
    train_by_user = {uid: g for uid, g in train_df[train_df['domain'] == domain].groupby('user_id')}

    candidates = [e for e in recs if has_valid_recs(e)]
    random.shuffle(candidates)

    sampled = []
    skipped_no_history = 0
    for entry in candidates:
        if len(sampled) >= n:
            break
        group = train_by_user.get(entry['user_id'])
        if group is None:
            skipped_no_history += 1
            continue
        history = get_valid_history(group)
        if history is None:
            skipped_no_history += 1
            continue
        sampled.append({
            'user_id': entry['user_id'],
            'domain': domain,
            'history': history,
            'recommended_item_ids': entry['recommended_item_ids'],
            'recommended_titles': entry['recommended_titles'],
        })

    print(f"{domain}: sampled {len(sampled)}/{n} requested users "
          f"({len(candidates)} had clean recs, {skipped_no_history} skipped for missing/thin history)")
    return sampled


sampled_book_users = sample_valid_users(als_books_recs, books_train, 'books')
sampled_movie_users = sample_valid_users(als_movies_recs, movies_train, 'movies')

sampled_book_users[0]  # sanity check

books: sampled 250/250 requested users (5000 had clean recs, 0 skipped for missing/thin history)
movies: sampled 250/250 requested users (1027 had clean recs, 0 skipped for missing/thin history)


{'user_id': 'AH7MWYZQBQPNOT75HQCE2M6KGVRQ',
 'domain': 'books',
 'history': [{'title': "The Negro's Civil War: How American Blacks Felt and Acted During the War for the Union",
   'rating': 5.0},
  {'title': 'The Great American Songbook - The Composers: Music and Lyrics for Over 100 Standards from the Golden Age of American Song',
   'rating': 5.0},
  {'title': 'Imperfect Harmony: Finding Happiness Singing with Others',
   'rating': 5.0},
  {'title': 'The Boy on the Porch', 'rating': 4.0},
  {'title': 'The Giver', 'rating': 3.0},
  {'title': 'The Death and Life of the Great American School System: How Testing and Choice Are Undermining Education',
   'rating': 5.0},
  {'title': 'The New Italian American Songbook', 'rating': 4.0},
  {'title': "Worlds of Music: An Introduction to the Music of the World's Peoples, Shorter Version",
   'rating': 4.0}],
 'recommended_item_ids': ['books::0736430512',
  'books::1250038820',
  'books::1476746583',
  'books::014240733X',
  'books::1451695195',


In [36]:
client = Anthropic(
    api_key=os.environ["ANTHROPIC_API_KEY"],
    base_url="https://openrouter.ai/api",
)

PROFILE_MODEL = "anthropic/claude-haiku-4.5"   # OpenRouter slug
MAX_WORKERS = 8
MAX_RETRIES = 5
RETRYABLE_ERRORS = (RateLimitError, APIConnectionError, InternalServerError, APITimeoutError)

# --- Structured output schema ---
PROFILE_TOOL = {
    "name": "submit_user_profile",
    "description": "Submit a synthetic user profile inferred from their rating history.",
    "input_schema": {
        "type": "object",
        "properties": {
            "description": {
                "type": "string",
                "description": "2-3 sentence persona summary of this reader/viewer."
            },
            "age": {"type": "integer", "description": "Plausible age estimate, 18-80."},
            "interests": {
                "type": "array",
                "items": {"type": "string"},
                "description": "3-6 general interests/hobbies suggested by their history."
            },
            "favorite_genres": {
                "type": "array",
                "items": {"type": "string"},
                "description": "2-5 genres/categories this person clearly favors."
            },
        },
        "required": ["description", "age", "interests", "favorite_genres"],
    },
}

PROFILE_SYSTEM = (
    "You infer realistic reader/viewer personas from a user's rating history. "
    "Base your inference only on the titles and ratings given. Be specific and concrete, "
    "not generic. Only items the user rated highly reflect genuine preference."
)


def call_claude_tool(model, system, user_message, tool, max_retries=MAX_RETRIES):
    """Call Claude, forcing a structured tool-call response. Retries only on transient errors."""
    last_err = None
    for attempt in range(max_retries):
        try:
            resp = client.messages.create(
                model=model,
                max_tokens=1024,
                system=system,
                messages=[{"role": "user", "content": user_message}],
                tools=[tool],
                tool_choice={"type": "tool", "name": tool["name"]},
            )
            for block in resp.content:
                if block.type == "tool_use":
                    return block.input
            raise ValueError(f"No tool_use block returned: {resp.content}")
        except RETRYABLE_ERRORS as e:
            last_err = e
            time.sleep(min(2 ** attempt + pyrandom.random(), 30))
        except Exception as e:
            # Non-transient (auth, bad request, etc.) - retrying won't help, fail fast with the real cause
            raise RuntimeError(f"Non-retryable error: {type(e).__name__}: {e}") from e

    raise RuntimeError(f"Failed after {max_retries} retries: {type(last_err).__name__}: {last_err}") from last_err


def build_profile_prompt(user_entry):
    domain = user_entry["domain"]
    lines = [f"- \"{h['title']}\" (rated {h['rating']}/5)" for h in user_entry["history"]]
    return (
        f"Here is a user's {domain} rating history, most recent first:\n"
        + "\n".join(lines)
        + f"\n\nInfer a plausible persona for this {domain} reader/viewer."
    )


def generate_profile(user_entry):
    prompt = build_profile_prompt(user_entry)
    profile = call_claude_tool(PROFILE_MODEL, PROFILE_SYSTEM, prompt, PROFILE_TOOL)
    return {**user_entry, "profile": profile}


def generate_profiles(sampled_users, save_path, max_workers=MAX_WORKERS, checkpoint_every=10):
    results = {}
    if os.path.exists(save_path):
        with open(save_path) as f:
            results = json.load(f)
        print(f"Resuming: {len(results)} profiles already saved at {save_path}")

    todo = [u for u in sampled_users if u["user_id"] not in results]
    print(f"{len(todo)} profiles to generate ({len(sampled_users) - len(todo)} already done)")

    with ThreadPoolExecutor(max_workers=max_workers) as pool:
        futures = {pool.submit(generate_profile, u): u for u in todo}
        done, failed = 0, 0
        start = time.time()
        for future in as_completed(futures):
            user_entry = futures[future]
            try:
                result = future.result()
                results[result["user_id"]] = result
            except Exception as e:
                failed += 1
                print(f"FAILED user_id={user_entry['user_id']}: {type(e).__name__}: {e}")
            done += 1
            if done % checkpoint_every == 0 or done == len(todo):
                with open(save_path, "w") as f:
                    json.dump(results, f, indent=2)
                elapsed = time.time() - start
                print(f"  {done}/{len(todo)} processed ({failed} failed), "
                      f"{elapsed:.0f}s elapsed, checkpointed")

    with open(save_path, "w") as f:
        json.dump(results, f, indent=2)
    return results


# --- Sanity check on ONE user before spending 400 calls ---
test_result = generate_profile(sampled_book_users[0])
print(test_result["profile"])

{'description': 'An intellectually engaged adult who combines scholarly interests in American history and social policy with a deep passion for music and vocal performance. This reader values both serious non-fiction that examines cultural and educational systems alongside practical guides to enriching personal experiences like group singing.', 'age': 55, 'interests': ['American history', 'vocal music and singing', 'music composition and standards', 'education policy', 'choral performance', 'cultural diversity in music'], 'favorite_genres': ['Non-fiction history', 'Music reference and instruction', 'Educational social commentary', 'American studies']}


In [37]:
book_profiles = generate_profiles(sampled_book_users, f'{DATA_DIR}/a-b-test/als_book_user_profiles.json')
movie_profiles = generate_profiles(sampled_movie_users, f'{DATA_DIR}/a-b-test/als_movie_user_profiles.json')

250 profiles to generate (0 already done)
  10/250 processed (0 failed), 4s elapsed, checkpointed
  20/250 processed (0 failed), 7s elapsed, checkpointed
  30/250 processed (0 failed), 9s elapsed, checkpointed
  40/250 processed (0 failed), 13s elapsed, checkpointed
  50/250 processed (0 failed), 16s elapsed, checkpointed
  60/250 processed (0 failed), 18s elapsed, checkpointed
  70/250 processed (0 failed), 21s elapsed, checkpointed
  80/250 processed (0 failed), 24s elapsed, checkpointed
  90/250 processed (0 failed), 26s elapsed, checkpointed
  100/250 processed (0 failed), 30s elapsed, checkpointed
  110/250 processed (0 failed), 33s elapsed, checkpointed
  120/250 processed (0 failed), 35s elapsed, checkpointed
  130/250 processed (0 failed), 39s elapsed, checkpointed
  140/250 processed (0 failed), 42s elapsed, checkpointed
  150/250 processed (0 failed), 45s elapsed, checkpointed
  160/250 processed (0 failed), 47s elapsed, checkpointed
  170/250 processed (0 failed), 51s elapse

## ALS: Evaluation of recommendations

In [38]:
import random as pyrandom

EVAL_MODEL = "anthropic/claude-haiku-4.5"

EVAL_TOOL = {
    "name": "submit_recommendation_evaluation",
    "description": "Submit engagement predictions for each recommended item, from this user's perspective.",
    "input_schema": {
        "type": "object",
        "properties": {
            "evaluations": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "item_id": {"type": "string", "description": "Echo back the exact item_id given."},
                        "will_view": {"type": "boolean", "description": "Would click into/view this item at all."},
                        "will_add_to_cart": {"type": "boolean", "description": "Would add to cart. Can only be true if will_view is true."},
                        "will_buy": {"type": "boolean", "description": "Would actually purchase. Can only be true if will_add_to_cart is true."},
                        "reason": {"type": "string", "description": "One short phrase, <=15 words."},
                    },
                    "required": ["item_id", "will_view", "will_add_to_cart", "will_buy", "reason"],
                },
            },
        },
        "required": ["evaluations"],
    },
}

EVAL_SYSTEM = (
    "You role-play as a specific person, using their profile and rating history, to predict "
    "realistic e-commerce behavior toward a list of recommended items. For each item, decide "
    "whether this specific person would view it, add it to cart, and buy it. "
    "These are a funnel: will_add_to_cart can only be true if will_view is true, and "
    "will_buy can only be true if will_add_to_cart is true. Most items should NOT be bought - "
    "be realistic and selective, the way a real shopper is."
)


def build_eval_prompt(record, shuffled_items):
    domain = record["domain"]
    p = record["profile"]
    history_lines = "\n".join(f"- \"{h['title']}\" (rated {h['rating']}/5)" for h in record["history"])
    items_lines = "\n".join(f"- item_id: {it['item_id']} | title: \"{it['title']}\"" for it in shuffled_items)

    return f"""User profile:
- Description: {p['description']}
- Age: {p['age']}
- Interests: {', '.join(p['interests'])}
- Favorite genres: {', '.join(p['favorite_genres'])}

This user's {domain} rating history:
{history_lines}

Here are 10 recommended {domain} items shown to this user (in random order, not ranked):
{items_lines}

For each item, predict whether this specific user would view it, add it to cart, and buy it."""


def evaluate_user(record):
    items = [
        {"item_id": iid, "title": title, "als_rank": rank + 1}
        for rank, (iid, title) in enumerate(zip(record["recommended_item_ids"], record["recommended_titles"]))
    ]
    shuffled = items.copy()
    pyrandom.shuffle(shuffled)

    prompt = build_eval_prompt(record, shuffled)
    output = call_claude_tool(EVAL_MODEL, EVAL_SYSTEM, prompt, EVAL_TOOL)

    als_rank_by_id = {it["item_id"]: it["als_rank"] for it in items}
    evaluations = []
    for e in output["evaluations"]:
        # Safety net: enforce funnel logic in code too, in case the model slips
        will_view = bool(e["will_view"])
        will_cart = bool(e["will_add_to_cart"]) and will_view
        will_buy = bool(e["will_buy"]) and will_cart
        evaluations.append({
            "item_id": e["item_id"],
            "als_rank": als_rank_by_id.get(e["item_id"]),
            "will_view": will_view,
            "will_add_to_cart": will_cart,
            "will_buy": will_buy,
            "reason": e.get("reason", ""),
        })

    return {
        "user_id": record["user_id"],
        "domain": record["domain"],
        "evaluations": evaluations,
        "n_view": sum(e["will_view"] for e in evaluations),
        "n_cart": sum(e["will_add_to_cart"] for e in evaluations),
        "n_buy": sum(e["will_buy"] for e in evaluations),
    }


def evaluate_users(records, save_path, max_workers=MAX_WORKERS, checkpoint_every=10):
    results = {}
    if os.path.exists(save_path):
        with open(save_path) as f:
            results = json.load(f)
        print(f"Resuming: {len(results)} evaluations already saved at {save_path}")

    todo = [r for r in records if r["user_id"] not in results]
    print(f"{len(todo)} evaluations to run ({len(records) - len(todo)} already done)")

    with ThreadPoolExecutor(max_workers=max_workers) as pool:
        futures = {pool.submit(evaluate_user, r): r for r in todo}
        done, failed = 0, 0
        start = time.time()
        for future in as_completed(futures):
            record = futures[future]
            try:
                result = future.result()
                results[result["user_id"]] = result
            except Exception as e:
                failed += 1
                print(f"FAILED user_id={record['user_id']}: {type(e).__name__}: {e}")
            done += 1
            if done % checkpoint_every == 0 or done == len(todo):
                with open(save_path, "w") as f:
                    json.dump(results, f, indent=2)
                elapsed = time.time() - start
                print(f"  {done}/{len(todo)} processed ({failed} failed), {elapsed:.0f}s elapsed, checkpointed")

    with open(save_path, "w") as f:
        json.dump(results, f, indent=2)
    return results


# --- Sanity check on ONE user first ---
test_eval = evaluate_user(list(book_profiles.values())[0])
test_eval

{'user_id': 'AFTOUCEL65IH3DJSJ3J62FQYPCBA',
 'domain': 'books',
 'evaluations': [{'item_id': 'books::B00Y21VGYW',
   'als_rank': 3,
   'will_view': False,
   'will_add_to_cart': False,
   'will_buy': False,
   'reason': 'Crime thriller lacks emotional depth this reader values'},
  {'item_id': 'books::B019G6DSDE',
   'als_rank': 6,
   'will_view': False,
   'will_add_to_cart': False,
   'will_buy': False,
   'reason': 'Serial killer thriller outside preferred memoir/poetry genres'},
  {'item_id': 'books::B016ZNRC0Q',
   'als_rank': 5,
   'will_view': False,
   'will_add_to_cart': False,
   'will_buy': False,
   'reason': 'Sensationalized thriller conflicts with authentic storytelling preference'},
  {'item_id': 'books::B00SLWQGUM',
   'als_rank': 10,
   'will_view': True,
   'will_add_to_cart': False,
   'will_buy': False,
   'reason': 'Psychological thriller; may view but likely too sensationalized overall'},
  {'item_id': 'books::B00YTXTIDO',
   'als_rank': 7,
   'will_view': False,
 

In [39]:
als_book_evals = evaluate_users(list(book_profiles.values()), f'{DATA_DIR}/a-b-test/als_book_evaluations.json')
als_movie_evals = evaluate_users(list(movie_profiles.values()), f'{DATA_DIR}/a-b-test/als_movie_evaluations.json')

250 evaluations to run (0 already done)
  10/250 processed (0 failed), 10s elapsed, checkpointed
  20/250 processed (0 failed), 15s elapsed, checkpointed
  30/250 processed (0 failed), 21s elapsed, checkpointed
  40/250 processed (0 failed), 27s elapsed, checkpointed
  50/250 processed (0 failed), 33s elapsed, checkpointed
  60/250 processed (0 failed), 39s elapsed, checkpointed
  70/250 processed (0 failed), 44s elapsed, checkpointed
  80/250 processed (0 failed), 50s elapsed, checkpointed
  90/250 processed (0 failed), 56s elapsed, checkpointed
  100/250 processed (0 failed), 61s elapsed, checkpointed
  110/250 processed (0 failed), 69s elapsed, checkpointed
  120/250 processed (0 failed), 74s elapsed, checkpointed
  130/250 processed (0 failed), 81s elapsed, checkpointed
  140/250 processed (0 failed), 87s elapsed, checkpointed
  150/250 processed (0 failed), 93s elapsed, checkpointed
  160/250 processed (0 failed), 98s elapsed, checkpointed
  170/250 processed (0 failed), 105s elap

## Bert: Building user profiles

In [47]:
def sample_valid_users(recs, train_df, domain, n=N_USERS_PER_DOMAIN, exclude_ids=None):
    """Randomly sample n users who have both clean recs and enough clean history,
    excluding any user_ids already used (e.g. by the ALS group)."""
    exclude_ids = exclude_ids or set()
    train_by_user = {uid: g for uid, g in train_df[train_df['domain'] == domain].groupby('user_id')}

    candidates = [e for e in recs if has_valid_recs(e) and e['user_id'] not in exclude_ids]
    pyrandom.shuffle(candidates)

    sampled = []
    skipped_no_history = 0
    for entry in candidates:
        if len(sampled) >= n:
            break
        group = train_by_user.get(entry['user_id'])
        if group is None:
            skipped_no_history += 1
            continue
        history = get_valid_history(group)
        if history is None:
            skipped_no_history += 1
            continue
        sampled.append({
            'user_id': entry['user_id'],
            'domain': domain,
            'history': history,
            'recommended_item_ids': entry['recommended_item_ids'],
            'recommended_titles': entry['recommended_titles'],
        })

    print(f"{domain}: sampled {len(sampled)}/{n} requested users "
          f"({len(candidates)} eligible after excluding {len(exclude_ids)} ALS users, "
          f"{skipped_no_history} skipped for missing/thin history)")
    return sampled


# Exclude exactly the users already used in the ALS group
als_book_user_ids = set(book_profiles.keys())
als_movie_user_ids = set(movie_profiles.keys())

sampled_bert_book_users = sample_valid_users(
    bert_books_recs, books_train, 'books', exclude_ids=als_book_user_ids
)
sampled_bert_movie_users = sample_valid_users(
    bert_movies_recs, movies_train, 'movies', exclude_ids=als_movie_user_ids
)

sampled_bert_book_users[0]  # sanity check

books: sampled 250/250 requested users (4855 eligible after excluding 250 ALS users, 0 skipped for missing/thin history)
movies: sampled 250/250 requested users (1147 eligible after excluding 250 ALS users, 0 skipped for missing/thin history)


{'user_id': 'AEZG46FY7RHQKHLLTHVO2QXM27OQ',
 'domain': 'books',
 'history': [{'title': 'The Year of Magical Thinking', 'rating': 4.0},
  {'title': "Aetius: Attila's Nemesis", 'rating': 2.0},
  {'title': "I'm Dying Up Here: Heartbreak and High Times in Stand-Up Comedy's Golden Era",
   'rating': 4.0},
  {'title': 'Conform: Exposing the Truth About Common Core and Public Education (The Control Series Book 2)',
   'rating': 5.0},
  {'title': 'The Story of Philosophy (Touchstone Books) (Touchstone Books (Paperback))',
   'rating': 5.0},
  {'title': 'The Righteous Mind: Why Good People Are Divided by Politics and Religion',
   'rating': 5.0},
  {'title': 'Team of Rivals: The Political Genius of Abraham Lincoln',
   'rating': 5.0},
  {'title': 'The Island of Lost Maps: A True Story of Cartographic Crime',
   'rating': 5.0},
  {'title': "Oh, the Places You'll Go!", 'rating': 5.0},
  {'title': 'The Greatest Minds and Ideas of All Time', 'rating': 2.0},
  {'title': 'Outlook 2007 For Dummies', '

In [48]:
bert_book_profiles = generate_profiles(sampled_bert_book_users, f'{DATA_DIR}/a-b-test/bert_book_user_profiles.json')
bert_movie_profiles = generate_profiles(sampled_bert_movie_users, f'{DATA_DIR}/a-b-test/bert_movie_user_profiles.json')

250 profiles to generate (0 already done)
  10/250 processed (0 failed), 5s elapsed, checkpointed
  20/250 processed (0 failed), 8s elapsed, checkpointed
  30/250 processed (0 failed), 10s elapsed, checkpointed
  40/250 processed (0 failed), 13s elapsed, checkpointed
  50/250 processed (0 failed), 16s elapsed, checkpointed
  60/250 processed (0 failed), 19s elapsed, checkpointed
  70/250 processed (0 failed), 23s elapsed, checkpointed
  80/250 processed (0 failed), 25s elapsed, checkpointed
  90/250 processed (0 failed), 28s elapsed, checkpointed
  100/250 processed (0 failed), 32s elapsed, checkpointed
  110/250 processed (0 failed), 34s elapsed, checkpointed
  120/250 processed (0 failed), 37s elapsed, checkpointed
  130/250 processed (0 failed), 40s elapsed, checkpointed
  140/250 processed (0 failed), 43s elapsed, checkpointed
  150/250 processed (0 failed), 45s elapsed, checkpointed
  160/250 processed (0 failed), 49s elapsed, checkpointed
  170/250 processed (0 failed), 51s elaps

## Bert: Evaluation of recommendations

In [50]:
bert_book_evals = evaluate_users(list(bert_book_profiles.values()), f'{DATA_DIR}/a-b-test/bert_book_evaluations.json')
bert_movie_evals = evaluate_users(list(bert_movie_profiles.values()), f'{DATA_DIR}/a-b-test/bert_movie_evaluations.json')

250 evaluations to run (0 already done)
  10/250 processed (0 failed), 9s elapsed, checkpointed
  20/250 processed (0 failed), 14s elapsed, checkpointed
  30/250 processed (0 failed), 19s elapsed, checkpointed
  40/250 processed (0 failed), 26s elapsed, checkpointed
  50/250 processed (0 failed), 32s elapsed, checkpointed
  60/250 processed (0 failed), 37s elapsed, checkpointed
  70/250 processed (0 failed), 44s elapsed, checkpointed
  80/250 processed (0 failed), 49s elapsed, checkpointed
  90/250 processed (0 failed), 54s elapsed, checkpointed
  100/250 processed (0 failed), 61s elapsed, checkpointed
  110/250 processed (0 failed), 67s elapsed, checkpointed
  120/250 processed (0 failed), 73s elapsed, checkpointed
  130/250 processed (0 failed), 78s elapsed, checkpointed
  140/250 processed (0 failed), 84s elapsed, checkpointed
  150/250 processed (0 failed), 91s elapsed, checkpointed
  160/250 processed (0 failed), 97s elapsed, checkpointed
  170/250 processed (0 failed), 102s elaps

## Comparing the results of ALS and BERT

In [5]:
def load_evals(path):
    with open(path) as f:
        return json.load(f)

DATA_DIR = '/content/drive/MyDrive/recsys-data'

als_book_evals = load_evals(f'{DATA_DIR}/a-b-test/als_book_evaluations.json')
als_movie_evals = load_evals(f'{DATA_DIR}/a-b-test/als_movie_evaluations.json')
bert_book_evals = load_evals(f'{DATA_DIR}/a-b-test/bert_book_evaluations.json')
bert_movie_evals = load_evals(f'{DATA_DIR}/a-b-test/bert_movie_evaluations.json')

In [6]:
def summarize_evaluations(evals_dict, domain, model_name, k=10):
    """Per-domain summary: average view/cart/buy rate out of k, across all users."""
    records = list(evals_dict.values())
    n = len(records)

    avg_view = sum(r["n_view"] for r in records) / n / k
    avg_cart = sum(r["n_cart"] for r in records) / n / k
    avg_buy = sum(r["n_buy"] for r in records) / n / k

    return {
        "model": model_name,
        "domain": domain,
        "n_users": n,
        "view_rate": round(avg_view, 4),
        "cart_rate": round(avg_cart, 4),
        "buy_rate": round(avg_buy, 4),
    }

In [7]:
als_books_summary = summarize_evaluations(als_book_evals, "books", "als")
als_movies_summary = summarize_evaluations(als_movie_evals, "movies", "als")

als_macro_summary = {
    "model": "als",
    "domain": "macro_avg",
    "n_users": als_books_summary["n_users"] + als_movies_summary["n_users"],
    "view_rate": round((als_books_summary["view_rate"] + als_movies_summary["view_rate"]) / 2, 4),
    "cart_rate": round((als_books_summary["cart_rate"] + als_movies_summary["cart_rate"]) / 2, 4),
    "buy_rate": round((als_books_summary["buy_rate"] + als_movies_summary["buy_rate"]) / 2, 4),
}

als_results_df = pd.DataFrame([als_books_summary, als_movies_summary, als_macro_summary])
als_results_df

,model,domain,n_users,view_rate,cart_rate,buy_rate
0,als,books,250,0.7552,0.5072,0.4624
1,als,movies,250,0.9264,0.6780,0.6312
2,als,macro_avg,500,0.8408,0.5926,0.5468


In [9]:
als_results_df.to_csv(f'{DATA_DIR}/a-b-test/als_a_b_results.csv', index=False)

In [8]:
bert_books_summary = summarize_evaluations(bert_book_evals, "books", "bert")
bert_movies_summary = summarize_evaluations(bert_movie_evals, "movies", "bert")

bert_macro_summary = {
    "model": "bert",
    "domain": "macro_avg",
    "n_users": bert_books_summary["n_users"] + bert_movies_summary["n_users"],
    "view_rate": round((bert_books_summary["view_rate"] + bert_movies_summary["view_rate"]) / 2, 4),
    "cart_rate": round((bert_books_summary["cart_rate"] + bert_movies_summary["cart_rate"]) / 2, 4),
    "buy_rate": round((bert_books_summary["buy_rate"] + bert_movies_summary["buy_rate"]) / 2, 4),
}

bert_results_df = pd.DataFrame([bert_books_summary, bert_movies_summary, bert_macro_summary])
bert_results_df

,model,domain,n_users,view_rate,cart_rate,buy_rate
0,bert,books,250,0.6972,0.4044,0.3608
1,bert,movies,250,0.8304,0.5520,0.5092
2,bert,macro_avg,500,0.7638,0.4782,0.4350


In [10]:
bert_results_df.to_csv(f'{DATA_DIR}/a-b-test/bert_a_b_results.csv', index=False)

## Running the Z-test

In [20]:
def user_level_rates(evals_dict, k=10):
    """One row per user: view/cart/buy rate as a fraction of k."""
    return pd.DataFrame([
        {
            "user_id": r["user_id"],
            "view_rate": r["n_view"] / k,
            "cart_rate": r["n_cart"] / k,
            "buy_rate": r["n_buy"] / k,
        }
        for r in evals_dict.values()
    ])

als_books_rates = user_level_rates(als_book_evals)
als_movies_rates = user_level_rates(als_movie_evals)
bert_books_rates = user_level_rates(bert_book_evals)
bert_movies_rates = user_level_rates(bert_movie_evals)
als_all_rates = pd.concat([als_books_rates, als_movies_rates], ignore_index=True)
bert_all_rates = pd.concat([bert_books_rates, bert_movies_rates], ignore_index=True)

comparisons = [
    ("macro_avg", "view_rate", als_all_rates, bert_all_rates),
    ("macro_avg", "cart_rate", als_all_rates, bert_all_rates),
    ("macro_avg", "buy_rate", als_all_rates, bert_all_rates),
    ("books", "view_rate", als_books_rates, bert_books_rates),
    ("books", "cart_rate", als_books_rates, bert_books_rates),
    ("books", "buy_rate", als_books_rates, bert_books_rates),
    ("movies", "view_rate", als_movies_rates, bert_movies_rates),
    ("movies", "cart_rate", als_movies_rates, bert_movies_rates),
    ("movies", "buy_rate", als_movies_rates, bert_movies_rates),
]

comparisons

[('macro_avg',
  'view_rate',
                            user_id  view_rate  cart_rate  buy_rate
  0    AHKI4KKDMH77MNWRY7CPBNC5X6LQ        0.9        0.8       0.8
  1    AH7MWYZQBQPNOT75HQCE2M6KGVRQ        0.8        0.4       0.4
  2    AFRCCKETSMZLGELP5IHGUERNL4LA        1.0        0.7       0.6
  3    AEPUTKX6FQJK7NLQB3XALHMMX2DA        0.4        0.2       0.2
  4    AFTOUCEL65IH3DJSJ3J62FQYPCBA        0.4        0.2       0.1
  ..                            ...        ...        ...       ...
  495  AF5G56E664Z5ODOMEF5CF7OJB7EQ        1.0        0.5       0.5
  496  AHIV5E4QZIRUS6OGUKXYJHKXBHKQ        0.9        0.4       0.3
  497  AEJ7RCG7E2R6D6NBYLCO3REWGSGA        1.0        0.9       0.8
  498  AGZA45U65OPRXR3JBFR4SUVQKEGA        0.9        0.7       0.7
  499  AFU6DS4YKWW6LQZIFDU2TPKE5KNA        1.0        0.7       0.7
  
  [500 rows x 4 columns],
                            user_id  view_rate  cart_rate  buy_rate
  0    AFGD5USVLWIUBGF5GFJLJ4UNH3VA        1.0        0.7

In [21]:
from scipy import stats
from statsmodels.stats.weightstats import ztest
from statsmodels.stats.multitest import multipletests

rows = []
for domain, metric, als_df, bert_df in comparisons:
    a, b = als_df[metric].values, bert_df[metric].values

    z_stat, p_z = ztest(a, b, alternative="two-sided")

    rows.append({
        "domain": domain, "metric": metric,
        "als_mean": round(a.mean(), 4), "bert_mean": round(b.mean(), 4),
        "diff": round(a.mean() - b.mean(), 4),
        "z_stat": round(z_stat, 3), "z_p": p_z,
    })

results_df = pd.DataFrame(rows)

results_df["significant_at_05"] = results_df["z_p"] < 0.05

results_df

,domain,metric,als_mean,bert_mean,diff,z_stat,z_p,significant_at_05
0,macro_avg,view_rate,0.8408,0.7638,0.0770,5.262,1.423727e-07,True
1,macro_avg,cart_rate,0.5926,0.4782,0.1144,7.468,8.168923e-14,True
2,macro_avg,buy_rate,0.5468,0.4350,0.1118,7.582,3.391740e-14,True
3,books,view_rate,0.7552,0.6972,0.0580,2.482,1.308227e-02,True
4,books,cart_rate,0.5072,0.4044,0.1028,4.697,2.633793e-06,True
5,books,buy_rate,0.4624,0.3608,0.1016,4.990,6.026164e-07,True
6,movies,view_rate,0.9264,0.8304,0.0960,6.511,7.488664e-11,True
7,movies,cart_rate,0.6780,0.5520,0.1260,6.646,3.006095e-11,True
8,movies,buy_rate,0.6312,0.5092,0.1220,6.471,9.724360e-11,True
